In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool
import pickle
import os
from torch_geometric.datasets import BA2MotifDataset
from torch.utils.data import Subset
from torch_geometric.data import Data
import networkx as ntx
from torch_geometric.utils import to_networkx,subgraph
import matplotlib.pyplot as plt

In [11]:
class GINGraphClf(nn.Module):
    def __init__(self, in_dim,out_dim, hidden_dim=64):
        super().__init__()
        self.conv1=GINConv(nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(),nn.BatchNorm1d(hidden_dim), nn.Linear(hidden_dim, hidden_dim),nn.ReLU()))
        self.conv2=GINConv(nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),nn.BatchNorm1d(hidden_dim), nn.Linear(hidden_dim, hidden_dim),nn.ReLU()))
        self.conv3=GINConv(nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),nn.BatchNorm1d(hidden_dim), nn.Linear(hidden_dim, hidden_dim),nn.ReLU()))
        
        self.mlp = nn.Sequential(
            nn.Linear(3*hidden_dim, hidden_dim//2),
            nn.Dropout(0.5),
            nn.ReLU(),
            nn.Linear(hidden_dim//2, out_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x, edge_index, batch):
        h1=self.conv1(x, edge_index)
        h2=self.conv2(h1, edge_index)
        h3=self.conv3(h2, edge_index)

        h1=global_add_pool(h1,batch)
        h2=global_add_pool(h2,batch)
        h3=global_add_pool(h3,batch)    


        x=torch.cat([h1,h2,h3],dim=1)


        return self.mlp(x)

In [12]:
dataset_path='./data/BA2Motif'
with open('./data/splits.pkl', 'rb') as f:
    splits=pickle.load(f)

full_dataset=BA2MotifDataset(root=dataset_path)
test_dataset=Subset(full_dataset,splits['test'])


test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)

In [13]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=GINGraphClf(test_dataset[0].num_features,2).to(device)


checkpoint=torch.load('./models/GIN/best_gin_model.pt',map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [14]:
model.eval()
for param in model.parameters():
    param.requires_grad = False

print(f"Model loaded from './models/best_gcn_model.pt'")
print(f"Best validation accuracy during training: {checkpoint['best_val_acc']:.4f}")

Model loaded from './models/best_gcn_model.pt'
Best validation accuracy during training: 1.0000


In [15]:
correct_indices = []
wrong_indices = []

def evaluate(model, loader, device):
    global_idx=0
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in loader:

            data = data.to(device)
            out = model(data.x, data.edge_index, data.batch)
            pred = out.argmax(dim=1)
            for i in range(len(pred)):
                if pred[i] == data.y[i]:
                    correct_indices.append({"index":global_idx,"pred":pred[i].item()})
                else:
                    wrong_indices.append({"index":global_idx,"pred":pred[i].item()})
                global_idx += 1
            correct += (pred == data.y).sum().item()
            total += data.num_graphs
    return correct / total

test_acc = evaluate(model, test_loader, device)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Number of correct predictions: {len(correct_indices)}")

Test Accuracy: 1.0000
Number of correct predictions: 100


In [16]:
print(f"correct indices: {correct_indices} , length: {len(correct_indices)}")
print(f"wrong indices: {wrong_indices}, length: {len(wrong_indices)}")
print(f"total: {len(correct_indices) + len(wrong_indices)}")

correct indices: [{'index': 0, 'pred': 0}, {'index': 1, 'pred': 1}, {'index': 2, 'pred': 0}, {'index': 3, 'pred': 0}, {'index': 4, 'pred': 0}, {'index': 5, 'pred': 1}, {'index': 6, 'pred': 0}, {'index': 7, 'pred': 1}, {'index': 8, 'pred': 1}, {'index': 9, 'pred': 0}, {'index': 10, 'pred': 1}, {'index': 11, 'pred': 1}, {'index': 12, 'pred': 1}, {'index': 13, 'pred': 0}, {'index': 14, 'pred': 0}, {'index': 15, 'pred': 0}, {'index': 16, 'pred': 1}, {'index': 17, 'pred': 0}, {'index': 18, 'pred': 0}, {'index': 19, 'pred': 1}, {'index': 20, 'pred': 1}, {'index': 21, 'pred': 1}, {'index': 22, 'pred': 1}, {'index': 23, 'pred': 0}, {'index': 24, 'pred': 0}, {'index': 25, 'pred': 0}, {'index': 26, 'pred': 1}, {'index': 27, 'pred': 1}, {'index': 28, 'pred': 1}, {'index': 29, 'pred': 1}, {'index': 30, 'pred': 0}, {'index': 31, 'pred': 0}, {'index': 32, 'pred': 1}, {'index': 33, 'pred': 1}, {'index': 34, 'pred': 0}, {'index': 35, 'pred': 1}, {'index': 36, 'pred': 1}, {'index': 37, 'pred': 1}, {'in

In [17]:
test_result_split={
    "correct": correct_indices,
    "wrong": wrong_indices
}
with open('./data/test_result.pkl','wb') as f:
    pickle.dump(test_result_split,f)

print("data indices saved to ./data/test_result.pkl")

data indices saved to ./data/test_result.pkl
